In [4]:
import lightgbm as lgb
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
)


# =========================================================
# 1. 데이터 불러오기
# =========================================================

current_path = Path.cwd()

project_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").exists()
)

data_path = (
    project_root
    / "data"
    / "preprocessing"
    / "train_processed.csv"
)

dataset = pd.read_csv(data_path, index_col=0)


# =========================================================
# 2. X / y 분리
# =========================================================

X = dataset.drop(columns=["Attrition"])
y = dataset["Attrition"]


# =========================================================
# 3. Train / Test 분리
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)


# =========================================================
# 4. LightGBM 기본 파라미터
# =========================================================

params = {
    "boosting_type": "gbdt",

    # 이진 분류
    "objective": "binary",

    # 평가 지표
    "metric": "binary_logloss",

    # 학습률
    "learning_rate": 0.05,

    # 하나의 트리가 가질 수 있는 최대 leaf 개수
    "num_leaves": 31,

    # -1 = 깊이 제한 없음
    "max_depth": -1,

    # 재현성
    "seed": 42,

    # 로그 최소화
    "verbosity": -1,
}


# =========================================================
# 5. LightGBM Dataset 생성
# =========================================================

train_dataset = lgb.Dataset(
    X_train,
    label=y_train,
)

test_dataset = lgb.Dataset(
    X_test,
    label=y_test,
    reference=train_dataset,
)


# =========================================================
# 6. 모델 학습
# =========================================================

gbm = lgb.train(
    params,
    train_dataset,
    valid_sets=[test_dataset],
    num_boost_round=100,
)


# =========================================================
# 7. 예측
# =========================================================

# Attrition=1일 확률
probabilities = gbm.predict(X_test)

# 0.5 이상이면 Attrition=1로 판단
predicted_labels = (probabilities >= 0.5).astype(int)


# =========================================================
# 8. 평가
# =========================================================

accuracy = accuracy_score(y_test, predicted_labels)
precision = precision_score(y_test, predicted_labels)
recall = recall_score(y_test, predicted_labels)
f1 = f1_score(y_test, predicted_labels)
roc_auc = roc_auc_score(y_test, probabilities)
average_precision = average_precision_score(y_test, probabilities)

print(f"Accuracy          : {accuracy:.4f}")
print(f"Precision         : {precision:.4f}")
print(f"Recall            : {recall:.4f}")
print(f"F1 Score          : {f1:.4f}")
print(f"ROC-AUC           : {roc_auc:.4f}")
print(f"Average Precision : {average_precision:.4f}")

print("\nClassification Report")
print(classification_report(y_test, predicted_labels))

Accuracy          : 0.7511
Precision         : 0.7371
Recall            : 0.7406
F1 Score          : 0.7389
ROC-AUC           : 0.8465
Average Precision : 0.8366

Classification Report
              precision    recall  f1-score   support

           0       0.76      0.76      0.76      6252
           1       0.74      0.74      0.74      5668

    accuracy                           0.75     11920
   macro avg       0.75      0.75      0.75     11920
weighted avg       0.75      0.75      0.75     11920



# 아래는 튜닝버전

In [13]:
import optuna
import lightgbm as lgb
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
)


# =========================================================
# 1. 데이터 불러오기
# =========================================================

current_path = Path.cwd()

project_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").exists()
)

data_path = (
    project_root
    / "data"
    / "preprocessing"
    / "train_processed.csv"
)

dataset = pd.read_csv(data_path, index_col=0)


# =========================================================
# 2. X / y 분리
# =========================================================

X = dataset.drop(columns=["Attrition"])
y = dataset["Attrition"]


# =========================================================
# 3. Train / Test 분리
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

def objective(trial):
    params = {
        "boosting_type": "gbdt",
        "objective": "binary",
        "metric": "binary_logloss",
        "n_estimators": trial.suggest_int("n_estimators",100,1000),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha",  1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", -1, 10),
        "random_state": 42,
        "verbosity": -1,
        "n_jobs": -1,

    }

    gbm = lgb.LGBMClassifier(**params)

    scores = cross_val_score(
        gbm,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,  # 사용 가능한 CPU 코어를 병렬로 활용
    )

    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50,timeout=600)  # 최대 50번의 시도 또는 10분 동안 최적화

trial = study.best_trial
print(f"Best value (ROC-AUC): {trial.value:.4f}")
print("Best hyperparameters: "  )
for key, value in trial.params.items():
    print(f"    {key}: {value}")
    


[I 2026-08-26 12:22:25,211] A new study created in memory with name: no-name-f545e86b-2699-4a88-b038-6118c063f9d6
[I 2026-08-26 12:22:35,772] Trial 0 finished with value: 0.8154205682374984 and parameters: {'n_estimators': 770, 'min_child_samples': 54, 'subsample': 0.9530113349283793, 'colsample_bytree': 0.8802161244767948, 'reg_alpha': 0.12442006527105678, 'reg_lambda': 1.2756192010257619e-06, 'learning_rate': 0.25091233287374554, 'num_leaves': 53, 'max_depth': 7}. Best is trial 0 with value: 0.8154205682374984.
[I 2026-08-26 12:22:39,110] Trial 1 finished with value: 0.8319101464770788 and parameters: {'n_estimators': 208, 'min_child_samples': 32, 'subsample': 0.784488692126491, 'colsample_bytree': 0.6954243594712819, 'reg_alpha': 1.0456121174005508e-05, 'reg_lambda': 0.004537991864499776, 'learning_rate': 0.2136015286987349, 'num_leaves': 69, 'max_depth': 8}. Best is trial 1 with value: 0.8319101464770788.
[I 2026-08-26 12:22:42,359] Trial 2 finished with value: 0.8469306044078297 a

Best value (ROC-AUC): 0.8517
Best hyperparameters: 
    n_estimators: 819
    min_child_samples: 70
    subsample: 0.6076254759192288
    colsample_bytree: 0.6167072238949133
    reg_alpha: 1.0250905315101452e-07
    reg_lambda: 0.0016015600239595435
    learning_rate: 0.12409117481975408
    num_leaves: 61
    max_depth: 1
